# SPX VIX Replication — ThetaData v3 Baseline

Version: v0.3  
Data source: ThetaData v3 terminal  
Goal: Replicate Cboe 30-day VIX using SPX / SPXW option quotes.

This notebook is the clean v3 baseline. It excludes diagnostic endpoint-discovery code.

In [3]:
# ============================================================
# Imports and global settings
# ============================================================

import io
import math
import requests
import numpy as np
import pandas as pd

from datetime import date, datetime


# ThetaData v3 local terminal endpoint
BASE_URL_V3 = "http://127.0.0.1:25503/v3"


# VIX calculation time: 4:00 PM ET
# Milliseconds after midnight
CALC_TIME_MS = 16 * 60 * 60 * 1000


# Temporary flat risk-free rate.
# We will improve this later with ThetaData rates.
DEFAULT_RISK_FREE_RATE = 0.05


# VIX methodology constants
MINUTES_PER_DAY = 1440
MINUTES_PER_YEAR = 525600
MINUTES_30_DAYS = 43200

In [4]:
# ============================================================
# Date and time helper functions
# ============================================================

def yyyymmdd_to_date(x):
    """
    Convert 20240116 to datetime.date(2024, 1, 16).
    """
    s = str(int(x))
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))


def yyyymmdd_to_dash_string(x):
    """
    Convert 20240116 to '2024-01-16'.
    """
    dt = yyyymmdd_to_date(x)
    return dt.strftime("%Y-%m-%d")


def ms_to_time_string(ms_of_day):
    """
    Convert milliseconds after midnight to ThetaData v3 time string.

    Example:
    57600000 -> '16:00:00.000'
    """
    total_seconds = int(ms_of_day // 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.000"


def is_third_friday(dt):
    """
    True if date is the standard monthly SPX expiration Friday.
    """
    return dt.weekday() == 4 and 15 <= dt.day <= 21

In [5]:
# ============================================================
# ThetaData v3 expiration loader
# ============================================================

def list_expirations_v3(symbol):
    """
    Get option expirations from ThetaData v3.

    v3 endpoint returns CSV text like:
        symbol,expiration
        "SPX","2024-02-16"

    This function converts expiration dates to YYYYMMDD integers.
    """
    url = BASE_URL_V3 + "/option/list/expirations"

    params = {
        "symbol": symbol
    }

    r = requests.get(url, params=params, timeout=60)

    print("URL:", r.url)
    print("Status code:", r.status_code)

    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    if df.empty:
        raise ValueError(f"No expirations returned for {symbol}")

    if "expiration" not in df.columns:
        raise ValueError(f"Could not find 'expiration' column. Columns are: {list(df.columns)}")

    expirations = (
        pd.to_datetime(df["expiration"])
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return sorted(expirations)


spx_exps = list_expirations_v3("SPX")
spxw_exps = list_expirations_v3("SPXW")

print()
print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))

print("SPX sample:", spx_exps[:5], "...", spx_exps[-5:])
print("SPXW sample:", spxw_exps[:5], "...", spxw_exps[-5:])

URL: http://127.0.0.1:25503/v3/option/list/expirations?symbol=SPX
Status code: 200
URL: http://127.0.0.1:25503/v3/option/list/expirations?symbol=SPXW
Status code: 200

SPX expirations loaded: 203
SPXW expirations loaded: 2202
SPX sample: [20120616, 20120721, 20120818, 20120922, 20121020] ... [20890221, 20890718, 20891219, 20900116, 20900618]
SPXW sample: [20120601, 20120608, 20120622, 20120706, 20120713] ... [20880920, 20880930, 20881018, 20890331, 20890630]


In [10]:
# ============================================================
# ThetaData v3 option-chain loader
# ============================================================

def get_chain_at_time(root, expi, trade_date, calc_time_ms, verbose=False):
    """
    Pull all strikes / both calls and puts for one SPX or SPXW expiration
    at a specific time using ThetaData v3.

    Returns dataframe with columns expected by the VIX math:
        root, expiration, strike, right, bid, ask, mid
    """

    request_time = ms_to_time_string(calc_time_ms)

    url = BASE_URL_V3 + "/option/history/quote"

    params = {
        "symbol": root,
        "expiration": yyyymmdd_to_dash_string(expi),
        "strike": "*",
        "right": "both",
        "start_date": yyyymmdd_to_dash_string(trade_date),
        "end_date": yyyymmdd_to_dash_string(trade_date),
        "start_time": request_time,
        "end_time": request_time,
        "interval": "1m",
        "format": "json"
    }

    r = requests.get(url, params=params, timeout=180)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)

    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(
            f"No data returned for {root} {expi} on {trade_date} at {request_time}"
        )

    df["root"] = df["symbol"]
    df["expiration"] = pd.to_datetime(df["expiration"]).dt.strftime("%Y%m%d").astype(int)

    right_map = {
        "CALL": "C",
        "PUT": "P",
        "C": "C",
        "P": "P"
    }

    df["right"] = df["right"].map(right_map)

    if df["right"].isna().any():
        bad_values = df["right"].unique()
        raise ValueError(f"Unknown option right values found: {bad_values}")

    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")

    df["mid"] = (df["bid"] + df["ask"]) / 2

    keep_cols = [
        "root",
        "expiration",
        "strike",
        "right",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "bid_exchange",
        "ask_exchange",
        "bid_condition",
        "ask_condition",
        "timestamp"
    ]

    df = df[keep_cols].copy()

    return df

In [7]:
# ============================================================
# Test v3 option-chain loader
# ============================================================

test_chain_v3 = get_chain_at_time(
    root="SPXW",
    expi=20240209,
    trade_date=20240116,
    calc_time_ms=CALC_TIME_MS
)

print()
print("Rows:", len(test_chain_v3))
print("Calls:", len(test_chain_v3[test_chain_v3["right"] == "C"]))
print("Puts:", len(test_chain_v3[test_chain_v3["right"] == "P"]))

display(test_chain_v3.head())

URL: http://127.0.0.1:25503/v3/option/history/quote?symbol=SPXW&expiration=2024-02-09&strike=%2A&right=both&start_date=2024-01-16&end_date=2024-01-16&start_time=16%3A00%3A00.000&end_time=16%3A00%3A00.000&interval=1m&format=json
Status code: 200

Rows: 340
Calls: 170
Puts: 170


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
0,SPXW,20240209,4350.0,C,429.1,437.7,433.4,15,15,5,5,50,50,2024-01-16T16:00:00.000
1,SPXW,20240209,4755.0,P,43.7,47.5,45.6,15,15,5,5,50,50,2024-01-16T16:00:00.000
2,SPXW,20240209,4755.0,C,69.5,74.5,72.0,15,15,5,5,50,50,2024-01-16T16:00:00.000
3,SPXW,20240209,4350.0,P,3.4,3.6,3.5,303,97,5,5,50,50,2024-01-16T16:00:00.000
4,SPXW,20240209,4840.0,P,84.2,90.4,87.3,15,15,5,5,50,50,2024-01-16T16:00:00.000


In [8]:
# ============================================================
# Core VIX math functions
# ============================================================

def preferred_root_for_expiration(exp_yyyymmdd):
    """
    Prefer SPX for standard monthly third-Friday expirations.
    Otherwise use SPXW.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def settlement_minutes_after_midnight_et(root):
    """
    Approximate VIX methodology settlement timing.

    SPX monthly options are AM-settled.
    SPXW weekly options are PM-settled.
    """
    if root == "SPX":
        return 9 * 60 + 30       # 9:30 AM ET

    if root == "SPXW":
        return 16 * 60           # 4:00 PM ET

    raise ValueError(f"Unknown root: {root}")


def minutes_to_expiry_vix_method(trade_date, exp_yyyymmdd, root, calc_time_ms):
    """
    Minutes from calculation time to option settlement time.
    """
    trade_dt = yyyymmdd_to_date(trade_date)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes_after_midnight = int(calc_time_ms // 60000)
    settlement_minutes = settlement_minutes_after_midnight_et(root)

    days_diff = (exp_dt - trade_dt).days

    return days_diff * MINUTES_PER_DAY + settlement_minutes - calc_minutes_after_midnight


def choose_vix_expirations_v3(trade_date, calc_time_ms=CALC_TIME_MS):
    """
    Select the two VIX-style expirations.

    Uses Friday expirations with roughly 23 to 37 days to expiry.
    Chooses the first two valid expirations.
    """
    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))

    candidates = []

    for exp in all_exps:
        exp_dt = yyyymmdd_to_date(exp)

        # Standard 30-day VIX uses Friday expirations.
        if exp_dt.weekday() != 4:
            continue

        root = preferred_root_for_expiration(exp)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms
        )

        days = minutes / MINUTES_PER_DAY

        if days > 23 and days < 37:
            candidates.append({
                "root": root,
                "expiration": exp,
                "minutes": minutes,
                "days": days
            })

    candidates = sorted(candidates, key=lambda x: x["minutes"])

    if len(candidates) < 2:
        raise ValueError(f"Could not find two valid VIX expirations for {trade_date}")

    return candidates[0], candidates[1]


def _prepare_call_put_tables(chain):
    """
    Create call and put tables indexed by strike.
    """
    df = chain.copy()

    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")

    df = df.dropna(subset=["strike", "bid", "ask", "mid", "right"])
    df = df[df["ask"] >= 0]
    df = df[df["bid"] >= 0]

    calls = (
        df[df["right"] == "C"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    puts = (
        df[df["right"] == "P"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    return calls, puts


def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Walk away from ATM.
    Keep options with positive bid.
    Stop after two consecutive zero-bid options.
    """
    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style variance for one expiration.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    # Infer forward using the strike where call/put mids are closest.
    parity_rows = []

    for K in common_strikes:
        call_mid = calls.loc[K, "mid"]
        put_mid = puts.loc[K, "mid"]
        diff = abs(call_mid - put_mid)

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "abs_call_put_diff": diff
        })

    parity_df = pd.DataFrame(parity_rows)
    min_row = parity_df.loc[parity_df["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    # OTM puts below K0, moving downward from K0.
    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    # OTM calls above K0, moving upward from K0.
    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    # K0 option uses average of call and put mid.
    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm
        ],
        ignore_index=True
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options
    }


def interpolate_to_30d(term_df):
    """
    Interpolate near and next term variances to 30 calendar days.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target = MINUTES_30_DAYS

    interpolated_variance = (
        T1 * var1 * ((N2 - target) / (N2 - N1))
        + T2 * var2 * ((target - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target)

    return 100 * math.sqrt(interpolated_variance)


def calculate_vix_for_date_v3(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    r=DEFAULT_RISK_FREE_RATE,
    return_raw_options=True
):
    """
    Calculate a replicated 30-day VIX value for one trade date.
    """
    near_exp, next_exp = choose_vix_expirations_v3(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_chain = get_chain_at_time(
        root=near_exp["root"],
        expi=near_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    next_chain = get_chain_at_time(
        root=next_exp["root"],
        expi=next_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_calc = calc_single_term_variance(
        chain=near_chain,
        minutes_to_expiry=near_exp["minutes"],
        r=r
    )

    next_calc = calc_single_term_variance(
        chain=next_chain,
        minutes_to_expiry=next_exp["minutes"],
        r=r
    )

    term_df = pd.DataFrame([
        {
            "term": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"],
            "variance": near_calc["variance"],
            "F": near_calc["F"],
            "K0": near_calc["K0"],
            "num_options": near_calc["num_options"]
        },
        {
            "term": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"],
            "variance": next_calc["variance"],
            "F": next_calc["F"],
            "K0": next_calc["K0"],
            "num_options": next_calc["num_options"]
        }
    ])

    vix = interpolate_to_30d(term_df)

    result = {
        "trade_date": trade_date,
        "vix": vix,
        "term_df": term_df,
        "near_calc": near_calc,
        "next_calc": next_calc
    }

    if return_raw_options:
        result["near_chain"] = near_chain
        result["next_chain"] = next_chain

    return result

In [11]:
# ============================================================
# One-date VIX validation
# ============================================================

result_20240116 = calculate_vix_for_date_v3(
    trade_date=20240116,
    calc_time_ms=CALC_TIME_MS,
    r=DEFAULT_RISK_FREE_RATE
)

print("Replicated VIX:")
print(result_20240116["vix"])

print("\nTerm details:")
display(result_20240116["term_df"])

Replicated VIX:
13.831907052437831

Term details:


,term,root,expiration,minutes,days,variance,F,K0,num_options
0,near,SPXW,20240209,34560,24.000000,0.018789,4781.504940,4780.0,161
1,next,SPX,20240216,44250,30.729167,0.019165,4783.694516,4780.0,351


In [12]:
# ============================================================
# Five-date validation against known Cboe VIX closes
# ============================================================

validation_dates = [
    {"trade_date": 20240116, "official_vix_close": 13.84},
    {"trade_date": 20240215, "official_vix_close": 14.01},
    {"trade_date": 20240315, "official_vix_close": 14.41},
    {"trade_date": 20240415, "official_vix_close": 19.23},
    {"trade_date": 20240515, "official_vix_close": 12.45},
]

validation_results = []

for row in validation_dates:
    trade_date = row["trade_date"]
    official = row["official_vix_close"]

    print(f"Calculating {trade_date}...")

    result = calculate_vix_for_date_v3(
        trade_date=trade_date,
        calc_time_ms=CALC_TIME_MS,
        r=DEFAULT_RISK_FREE_RATE,
        return_raw_options=False
    )

    replicated = result["vix"]

    validation_results.append({
        "trade_date": trade_date,
        "replicated_vix": replicated,
        "official_vix_close": official,
        "diff": replicated - official,
        "abs_diff": abs(replicated - official)
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Average absolute difference:", validation_df["abs_diff"].mean())
print("Max absolute difference:", validation_df["abs_diff"].max())

Calculating 20240116...
Calculating 20240215...
Calculating 20240315...
Calculating 20240415...
Calculating 20240515...


,trade_date,replicated_vix,official_vix_close,diff,abs_diff
0,20240116,13.831907,13.84,-0.008093,0.008093
1,20240215,14.175246,14.01,0.165246,0.165246
2,20240315,14.533669,14.41,0.123669,0.123669
3,20240415,19.164770,19.23,-0.065230,0.065230
4,20240515,12.389784,12.45,-0.060216,0.060216


Average absolute difference: 0.08449088289750258
Max absolute difference: 0.16524625690045092


## v0.3 v3 baseline validation

The ThetaData v3 migration is complete.

Five-date validation:

| Date | Replicated VIX | Official VIX Close | Difference |
|---|---:|---:|---:|
| 2024-01-16 | 13.8319 | 13.84 | -0.0081 |
| 2024-02-15 | 14.1752 | 14.01 | 0.1652 |
| 2024-03-15 | 14.5337 | 14.41 | 0.1237 |
| 2024-04-15 | 19.1648 | 19.23 | -0.0652 |
| 2024-05-15 | 12.3898 | 12.45 | -0.0602 |

Average absolute difference: 0.0845 VIX points  
Max absolute difference: 0.1652 VIX points

This notebook is now the clean v3 baseline. Future improvements should be made in copied versions, not directly in this file.